In [ ]:
! pip install opencv-python tqdm
import os
import csv
import subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import cv2

# Video clipping

In [ ]:
def read_csv_to_list(file_path):
    """
    Read a CSV file and return a 2D list (list of lists)
    :param file_path: Path to CSV file
    :return: List of rows
    """
    data_list = []
    with open(file_path, mode='r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for row in reader:
            # Convert each element to int
            data_list.append([int(v) for v in row])
    return data_list


In [ ]:
import os
import cv2
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

def process_folder(args):
    folder_path, label_rows, save_dir, video_root = args
    video_file = os.path.join(video_root, folder_path, f"{folder_path}.mp4")
    if not os.path.isfile(video_file):
        print(f"[Warning] Video not found: {video_file}")
        return

    cap = cv2.VideoCapture(video_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    for ids, (cls, start_frame, end_frame) in enumerate(label_rows):
        prefix = f"{folder_path}"
        out_name = f"{prefix}_{ids}.mp4"
        out_path = os.path.join(save_dir, out_name)

        cap = cv2.VideoCapture(video_file)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        for frame_idx in range(start_frame, end_frame + 1):
            ret, frame = cap.read()
            if not ret:
                break
            writer.write(frame)

        cap.release()
        writer.release()
        # print(f"[Info] Slice saved: {out_path}")


In [ ]:
dataset_root = '../MiGA/imigue_data_phase1/datasets/imigue_skeleton_train'
# Read all labels
all_data = {}
for entry in os.listdir(dataset_root):
    fp = os.path.join(dataset_root, entry)
    if os.path.isdir(fp):
        csv_file = os.path.join(fp, f"{entry}_label.csv")
        if os.path.isfile(csv_file):
            all_data[entry] = read_csv_to_list(csv_file)

# Parameter setup
video_root = '../MiGA/imigue_rgb_phase1/train_data'
save_folder = '../RGB/train'
os.makedirs(save_folder, exist_ok=True)

# Combine parameters into a list of argument tuples
task_args = [(folder, rows, save_folder, video_root) for folder, rows in all_data.items()]
# Multiprocessing
num_workers = min(cpu_count(), 8)  # Use up to 64 or all available cores
with Pool(num_workers) as pool:
    list(tqdm(pool.imap_unordered(process_folder, task_args), total=len(task_args)))

In [ ]:
dataset_root = '../MiGA/imigue_data_phase1/datasets/imigue_skeleton_validate'
# Read all labels
all_data = {}
for entry in os.listdir(dataset_root):
    fp = os.path.join(dataset_root, entry)
    if os.path.isdir(fp):
        csv_file = os.path.join(fp, f"{entry}_label.csv")
        if os.path.isfile(csv_file):
            all_data[entry] = read_csv_to_list(csv_file)

# Parameter setup
video_root = '../MiGA/imigue_rgb_phase1/validation_data'
save_folder = '../RGB/val'
os.makedirs(save_folder, exist_ok=True)

# Combine parameters into a list of argument tuples
task_args = [(folder, rows, save_folder, video_root) for folder, rows in all_data.items()]
# Multiprocessing
num_workers = min(cpu_count(), 8)  # Use up to 64 or all available cores
with Pool(num_workers) as pool:
    list(tqdm(pool.imap_unordered(process_folder, task_args), total=len(task_args)))

In [ ]:
dataset_root = '../MiGA/imigue_data_phase2/imigue_skeleton_test'
# Read all labels
all_data = {}
for entry in os.listdir(dataset_root):
    fp = os.path.join(dataset_root, entry)
    if os.path.isdir(fp):
        csv_file = os.path.join(fp, f"{entry}_label.csv")
        if os.path.isfile(csv_file):
            all_data[entry] = read_csv_to_list(csv_file)

# Parameter setup
video_root = '../MiGA/imigue_rgb_phase2'
save_folder = '../RGB/test'
os.makedirs(save_folder, exist_ok=True)

# Combine parameters into a list of argument tuples
task_args = [(folder, rows, save_folder, video_root) for folder, rows in all_data.items()]
# Multiprocessing
num_workers = min(cpu_count(), 8)  # Use up to 64 or all available cores
with Pool(num_workers) as pool:
    list(tqdm(pool.imap_unordered(process_folder, task_args), total=len(task_args)))